# PC Lab 2: Distances, Preprocessing and Nearest Neighbors

In the previous lab you explored a dataset. In this one you will build your first
predictive models, and you will build them twice: once by hand with NumPy, so that you
know exactly what the algorithm does, and once with scikit-learn, which is what you will
use in practice.

### What you will be able to do after this lab

1. Compute distances between instances and find an instance's nearest neighbors.
2. Implement $k$-nearest neighbors from scratch, for **classification** (a vote) and for
   **regression** (an average), and explain what $k$ controls.
3. Read a decision boundary, and recognize under- and overfitting in one.
4. Prepare data for a distance-based model: dummy encoding and standardization.
5. Evaluate a model honestly: a held-out test set, a sensible baseline, and an error
   metric you can interpret.


---

The code below downloads the three data files used in this lab. Run it first.

In [ ]:
import urllib.request

for f in ["iris120.csv", "irisNA.csv", "abalone.csv"]:
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/BioML-UGent/MLLS/main/02_knn/{f}", f)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixing the seed makes every random choice below reproducible: you, your neighbor and
# the demonstrator all get the same numbers. Do this in every analysis you report.
RANDOM_STATE = 42

## 1. Nearest neighbor classification

### The classification problem

In the previous lab session the iris flower dataset was explored. Imagine now that for a new
iris flower we know its measurements (sepal length and sepal width), but not its species. A
natural task is to assign the flower to one of the three species based on those measurements.
This is a **classification problem**.

### The dataset

In `iris120.csv` each instance (flower) is described by 5 attributes: sepal length, sepal
width, petal length, petal width and species. For most of this lab we use only the species
and the two *sepal* measurements, not because two features are better, but because two
features can be drawn on a page. We will come back to that choice in section 3.

These attributes are *variables*. In a classification setting the aim is to predict one of
them (here the species) from the others (here sepal length and width). The variable to be
predicted is the **output variable**; the variables used to predict it are the **input
variables** or **features**.

A dataset $\mathcal{D}$ consists of $n$ observations of input-output pairs
$(\boldsymbol{x}_i, y_i)$ with $i \in \{1, \dots, n\}$. Here
$\boldsymbol{x}_i = (x_{i1}, \dots, x_{ip})^T \in \mathbb{R}^p$ holds the observed feature
values ($p$ = number of features) and $y_i$ is the observed output, so that a training set
with $n$ instances is

$$\mathcal{D}_{train} = \{ (\boldsymbol{x}_1, y_1), \dots, (\boldsymbol{x}_n, y_n) \}.$$

Throughout the lab, **the first index counts instances and the second counts features**:
$x_{ij}$ is feature $j$ of instance $i$.

### Model setup

The goal is to build a *model* $f$ that predicts the output from the inputs. In our problem
both inputs are real-valued ($\boldsymbol{x}_i \in \mathbb{R}^2$) and the output is nominal,
taking values in $\{\textit{setosa}, \textit{versicolor}, \textit{virginica}\}$. So the model
we are looking for is a mapping

$$f: \mathbb{R}^2 \rightarrow \{\textit{setosa}, \textit{versicolor}, \textit{virginica}\}.$$

### The nearest neighbor algorithm

A very simple way to derive a classifier from a training set is the **nearest neighbor
algorithm**. It rests on one assumption: *instances with similar features are likely to have
the same label.*

To decide which training instances are "closest", we need a measure of distance
$d(\boldsymbol{x}_i, \boldsymbol{x}_j)$. We will use the Euclidean distance:

$$ d_E(\boldsymbol{x}_a, \boldsymbol{x}_b) = \sqrt{\sum_{j=1}^{p} (x_{aj} - x_{bj})^2}. $$

The 1-nearest-neighbor algorithm is then just two steps:

1. For an instance with unknown label and known feature vector $\boldsymbol{x}$, compute the
   distance to every instance in the training set: $d_E(\boldsymbol{x}, \boldsymbol{x}_i)$
   for $i = 1, \dots, n$.
2. Take the label of the closest training instance as the prediction.

<div class="alert alert-success">

<b>EXERCISE 1.1</b>: <b>Load <code>iris120.csv</code> into memory and select the columns
'Sepal.Length', 'Sepal.Width' and 'Species'. Do the same for the set of unlabeled flowers
(<code>irisNA.csv</code>). Both should end up as pandas DataFrames.</b>

Replace every <code>...</code> with your own code.
</div>

In [ ]:
cols = ['Sepal.Length', 'Sepal.Width', 'Species']

# iris120: our labeled training data
iris120 = ...   # TODO: read iris120.csv into a DataFrame
iris120 = ...   # TODO: keep only the columns in `cols`

# irisNA: flowers whose species we do not know
irisNA = ...    # TODO: read irisNA.csv into a DataFrame
irisNA = ...    # TODO: keep only the columns in `cols`

iris120.head()

Notice that the `Species` column of `irisNA` is entirely `NA`. Those are **unknown
labels**, the thing we want to predict.

Before modeling anything, look at the data.

In [ ]:
SPECIES = ['setosa', 'versicolor', 'virginica']
COLORS = {'setosa': '#4C72B0', 'versicolor': '#DD8452', 'virginica': '#55A868'}

fig, ax = plt.subplots(figsize=(6, 5))
for sp in SPECIES:
    sub = iris120[iris120['Species'] == sp]
    ax.scatter(sub['Sepal.Length'], sub['Sepal.Width'],
               c=COLORS[sp], label=sp, s=40, edgecolor='white', linewidth=0.5)
ax.set_xlabel("sepal length (cm)")
ax.set_ylabel("sepal width (cm)")
ax.set_title("The 120 labeled training flowers")
ax.legend()
plt.show()

print(iris120['Species'].value_counts())

<div class="alert alert-info">

<b>Think about it.</b>

1. Which of the three species looks easy to identify from these two measurements alone, and
   which two do you expect a classifier to confuse?
2. Roughly where on this plot would a 1-nearest-neighbor classifier make its mistakes?
3. If you could measure one more thing about each flower to help, what would you ask for?
</div>

**Your answer:**




<div class="alert alert-success">

<b>EXERCISE 1.2</b>: <b>Implement the nearest neighbor algorithm for the iris problem in a
function <code>nn_iris_predict</code>, following the skeleton below.</b>

Here <code>new_obs_features</code> is an array holding the two features of one new instance,
and <code>train_dataset</code> is the DataFrame with features and labels (i.e. <code>iris120</code>).
</div>

In [ ]:
def nn_iris_predict(new_obs_features, train_dataset):
    # features of the training instances (first two columns), as a plain NumPy array
    train_features = ...   # TODO

    # labels of the training instances
    train_labels = ...     # TODO

    # Euclidean distance from the new instance to every training instance
    # (hint: you can do this in one line, without a loop)
    dist_euc = ...         # TODO

    # position of the smallest distance (hint: np.argmin)
    nn_ind = ...           # TODO

    # the label at that position
    nn_label = ...         # TODO

    return nn_label

Test it on the first unlabeled flower.

In [ ]:
new_obs_features = irisNA.iloc[0, :2].values
prediction = nn_iris_predict(new_obs_features, iris120)

print(f"features {new_obs_features} -> predicted species: {prediction}")


The prediction is nothing more than "whichever training flower sits closest on the
scatter plot". Let us draw exactly that.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for sp in SPECIES:
    sub = iris120[iris120['Species'] == sp]
    ax.scatter(sub['Sepal.Length'], sub['Sepal.Width'],
               c=COLORS[sp], label=sp, s=40, edgecolor='white', linewidth=0.5)

# the new, unlabeled flower
ax.scatter(*new_obs_features, marker='*', s=220, c='black',
           label='new flower', zorder=5)

# draw a line to its nearest neighbor
train_xy = iris120.iloc[:, :2].values
d = np.sqrt(((train_xy - new_obs_features) ** 2).sum(axis=1))
nn = train_xy[np.argmin(d)]
ax.plot([new_obs_features[0], nn[0]], [new_obs_features[1], nn[1]],
        'k-', linewidth=1.5, zorder=4)
ax.scatter(*nn, marker='o', s=140, facecolor='none', edgecolor='black',
           linewidth=1.5, label='nearest neighbor', zorder=6)
ax.add_patch(plt.Circle(new_obs_features, d.min(), fill=False,
                        linestyle=':', edgecolor='black', linewidth=1))

ax.set_xlabel("sepal length (cm)")
ax.set_ylabel("sepal width (cm)")
ax.set_title(f"1-NN: nearest training flower is {prediction}")
ax.legend(loc='upper right')
plt.show()

## 2. From 1-NN to $k$-NN

Trusting a single neighbor is fragile: one mislabeled or unusual training flower is enough
to flip the prediction. The obvious fix is to ask several neighbors and let them vote.

Let $N_k(\boldsymbol{x}) \subset \mathcal{D}$ be the set of $k$ training instances closest to
$\boldsymbol{x}$. Then:

- for **classification**, $f(\boldsymbol{x})$ is the *plurality vote* of $N_k(\boldsymbol{x})$,
  the label occurring most often among the $k$ neighbors;
- for **regression** (section 5), $f(\boldsymbol{x})$ is the *average* of their outputs.

Setting $k = 1$ recovers the algorithm you just wrote.

**A wrinkle: ties.** With $k = 2$ and two neighbors of different species, or $k = 3$ split
across three species, the vote has no winner. Implementations must break the tie somehow, and
they differ in how. scikit-learn picks the class that comes first in sorted order, an
arbitrary but deterministic rule. This is why an odd $k$ is conventional for two-class
problems: it makes ties impossible.

<div class="alert alert-success">

<b>EXERCISE 2.1</b>: <b>Generalize your function to <code>knn_iris_predict(new_obs_features,
train_dataset, k)</code>, which returns the plurality vote of the <code>k</code> nearest
neighbors.</b>

Two hints: <code>np.argsort</code> returns the positions that would sort an array, so the
first <code>k</code> entries are the <code>k</code> nearest neighbors. And
<code>pd.Series(labels).mode()[0]</code> gives you the most frequent label (breaking ties
alphabetically, just like scikit-learn).
</div>

In [ ]:
def knn_iris_predict(new_obs_features, train_dataset, k=1):
    train_features = ...   # TODO (same as before)
    train_labels = ...     # TODO (same as before)

    dist_euc = ...         # TODO (same as before)

    # positions of the k smallest distances (hint: np.argsort)
    nn_ind = ...           # TODO

    # the labels of those k neighbors
    nn_labels = ...        # TODO

    # plurality vote among nn_labels
    return ...             # TODO

Try your function out: apply it to all 30 unlabeled flowers and count how often each species
comes out. If your vote is implemented correctly you should see all three species represented.

In [ ]:
# one prediction per unlabeled flower, using your own function with k = 5
preds_k5 = [knn_iris_predict(x, iris120, k=5) for x in irisNA.iloc[:, :2].values]
print(pd.Series(preds_k5).value_counts())

### What $k$ actually does

A classifier partitions the feature space into regions, one per predicted class. The borders
between those regions are the **decision boundary**. Because we kept only two features, we can
draw it: evaluate the model on a fine grid of points and color each one by its prediction.

Look at what happens as $k$ grows.

In [ ]:
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier

X_iris = iris120[['Sepal.Length', 'Sepal.Width']].values
y_iris = iris120['Species'].values
X_new = irisNA[['Sepal.Length', 'Sepal.Width']].values

cmap = ListedColormap([COLORS[s] for s in SPECIES])

# a grid covering the feature space
xx, yy = np.meshgrid(np.linspace(X_iris[:, 0].min() - .3, X_iris[:, 0].max() + .3, 300),
                     np.linspace(X_iris[:, 1].min() - .3, X_iris[:, 1].max() + .3, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

ks = [1, 5, 25, 120]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), sharex=True, sharey=True)
for ax, k in zip(axes, ks):
    clf = KNeighborsClassifier(n_neighbors=k).fit(X_iris, y_iris)
    Z = np.searchsorted(SPECIES, clf.predict(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=.25, cmap=cmap, levels=[-.5, .5, 1.5, 2.5])
    for sp in SPECIES:
        sub = iris120[iris120['Species'] == sp]
        ax.scatter(sub['Sepal.Length'], sub['Sepal.Width'],
                   c=COLORS[sp], s=22, edgecolor='white', linewidth=0.4)
    ax.set_title(f"k = {k}" + ("   (= n)" if k == 120 else ""))
    ax.set_xlabel("sepal length (cm)")
axes[0].set_ylabel("sepal width (cm)")
handles = [plt.Line2D([], [], marker='o', linestyle='', color=COLORS[s], label=s)
           for s in SPECIES]
axes[0].legend(handles=handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

<div class="alert alert-info">

<b>Think about it.</b>

1. At $k = 1$ the boundary is jagged and there are small islands of one color stranded inside
   another. What produces an island, and would you expect a new flower landing in one to really
   be that species?
2. At $k = 120$ the whole plane is a single color. Which one, and why?
3. Both extremes are bad, but for opposite reasons. Describe what is going wrong in each panel,
   and say which of the two models you would trust less on a flower it has never seen. Do you
   know names for these two failure modes?
</div>

**Your answer:**




## 3. Does it actually work? Evaluating the classifier

We have predictions for the 30 flowers in `irisNA`, but their true species are unknown, so
those predictions cannot tell us whether the model is any good. To measure performance we need
data where we know the answer but the model does not.

The standard device is to **split the labeled data**: train on one part, and evaluate on the
other, which the model never sees during fitting. For classification the simplest metric is
**accuracy**, the fraction of test instances predicted correctly.

<div class="alert alert-success">

<b>EXERCISE 3.1</b>: <b>Split <code>iris120</code> into 75% training and 25% test data, fit a
5-nearest-neighbor classifier on the training part, and report accuracy on both parts.</b>

Use scikit-learn's
<a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html"><code>train_test_split</code></a>,
and pass <code>random_state=RANDOM_STATE</code> and <code>stratify=y_iris</code> (the latter keeps
the class proportions the same in both halves).
</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_tr, X_te, y_tr, y_te = ...   # TODO: 75/25 split, random_state=RANDOM_STATE, stratify=y_iris

clf = ...                      # TODO: instantiate a 5-NN classifier and fit it on the training part

acc_train = ...                # TODO
acc_test = ...                 # TODO

print(f"training accuracy: {acc_train:.3f}")
print(f"test accuracy:     {acc_test:.3f}")

Test accuracy is well below training accuracy: the model looks better on the data it was
fitted to than on data it has not seen. That gap is the whole reason we hold out a test set.

But is 70% good? Compare it to the laziest possible model: always predict the most common
species. And remember the scatter plot: we are asking two overlapping features to do a job
they cannot do. Let us check what the other two measurements buy us.

In [ ]:
from sklearn.dummy import DummyClassifier

# baseline: always predict the majority class
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
print(f"majority-class baseline: {accuracy_score(y_te, dummy.predict(X_te)):.3f}")

# the same model, but using all four measurements
iris_full = pd.read_csv("iris120.csv")
X4 = iris_full[['Sepal.Length', 'Sepal.Width', 'Petal.Length', 'Petal.Width']].values
X4_tr, X4_te, y4_tr, y4_te = train_test_split(
    X4, y_iris, test_size=0.25, random_state=RANDOM_STATE, stratify=y_iris)
clf4 = KNeighborsClassifier(n_neighbors=5).fit(X4_tr, y4_tr)
print(f"5-NN on 2 sepal features:  {acc_test:.3f}")
print(f"5-NN on all 4 features:    {accuracy_score(y4_te, clf4.predict(X4_te)):.3f}")

Adding the petal measurements takes test accuracy from 0.70 to 0.97. The limitation was
never the algorithm. It was the information we gave it.

(This does not mean more features are always better. In very high dimensions the opposite
problem appears: distances concentrate, every point looks equally far away, and "nearest
neighbor" stops meaning much.)

Now that we have a model we trust, we can finally use it for what we set out to do: label the
unknown flowers.

In [ ]:
irisNA_pred = clf.predict(X_new)
result = irisNA[['Sepal.Length', 'Sepal.Width']].copy()
result['predicted species'] = irisNA_pred
print(result.head(10))
print("\npredicted composition:")
print(pd.Series(irisNA_pred).value_counts())

## 4. Preparing data for a distance-based model

Everything $k$-NN does flows from one number: the distance between two instances. That makes it
unusually sensitive to how the features are *expressed*. Two issues come up in almost every
real dataset.

For this section we switch to a new problem: predicting the age of abalone, a marine snail,
from physical measurements.

<img src="https://upload.wikimedia.org/wikipedia/commons/3/33/LivingAbalone.JPG?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original" width=500>

`abalone.csv` holds measurements of 700 specimens (a subsample of the
[4177 in the original dataset](https://archive.ics.uci.edu/dataset/1/abalone)). Determining an
abalone's age normally means cutting the shell, staining it and counting rings under a
microscope, tedious enough that predicting it from cheap external measurements is genuinely
useful.

In [ ]:
abalone = pd.read_csv("abalone.csv")
print(abalone.head())
print("\nshape:", abalone.shape)
print(abalone['sex'].value_counts())

<div class="alert alert-info">
<b>Know what is in your columns.</b> Two things worth noticing before modeling anything.

<b>The <code>age</code> column is a ring count</b>, not a number of years; the dataset's
documentation defines age in years as rings $+\ 1.5$. Our models predict rings. That is a
monotone shift, so it changes nothing about the modeling, but reporting "this snail is 9
years old" when you predicted 9 rings would be wrong by 18 months.

<b>The <code>sex</code> column has three values</b>, <code>M</code>, <code>F</code> and
<code>I</code> for <i>infant</i>. That last one is a developmental stage rather than a sex, so this
feature silently encodes immaturity, which is strongly related to age. Useful for prediction,
but worth knowing it is there.
</div>

### 4.1 Dummy encoding of nominal features

Distances need numbers, and `sex` is a name. Encoding it as $M = 0, F = 1, I = 2$ would be
worse than useless: it would assert that F sits halfway between M and I.

Instead, a nominal feature taking $k$ possible values is replaced by $k$ new binary features,
one per value. This is called a **dummy encoding** (or *one-hot* encoding). For a weather feature $x^1$ taking
values $\{\textit{Sunny}, \textit{Overcast}, \textit{Rainy}\}$ we create $x^{1a}, x^{1b}, x^{1c}$
with

$$
x^{1a} = \begin{cases} 1 & \text{if } x^1 = \text{Sunny} \\ 0 & \text{otherwise} \end{cases}
\qquad
x^{1b} = \begin{cases} 1 & \text{if } x^1 = \text{Overcast} \\ 0 & \text{otherwise} \end{cases}
\qquad
x^{1c} = \begin{cases} 1 & \text{if } x^1 = \text{Rainy} \\ 0 & \text{otherwise} \end{cases}
$$

pandas' [`get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)
does this in one call. (scikit-learn's
[`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
is the equivalent for use inside a pipeline.)

In [ ]:
abalone_encoded = pd.get_dummies(abalone, columns=['sex'])
print(abalone_encoded.head())

The three new columns are named `sex_F`, `sex_I`, `sex_M`.

One column here is redundant: knowing two of them tells you the third. `get_dummies(...,
drop_first=True)` removes it. For $k$-NN it makes no real difference, but for the linear models
in lab 3 that redundancy causes an actual problem, so you will meet the argument again there.

### 4.2 Standardization, and why units matter

Real features have different means and spreads. In a Euclidean distance, a feature with a large
numerical range contributes larger differences and therefore dominates the distance, regardless
of whether it is the more informative feature.

The usual remedy is **standardization**: center each feature on its mean and divide by its
standard deviation,

$$ x^{j\prime} = \frac{x^j - \mu_j}{\sigma_j}, $$

with $\mu_j$ and $\sigma_j$ the sample mean and standard deviation of feature $j$. Afterwards
every feature has mean 0 and standard deviation 1, so each contributes comparably.

That is the standard argument. Rather than take it on faith, let us make the problem visible.

Rather than take that on faith, watch it happen. The cell below re-expresses sepal length in
millimeters instead of centimeters and re-runs 1-NN. Nothing about the flowers themselves has
changed.

In [ ]:
# the same flowers, with sepal length expressed in millimeters
X_iris_mm = X_iris.copy(); X_iris_mm[:, 0] *= 10
X_new_mm = X_new.copy();  X_new_mm[:, 0] *= 10

pred_cm = KNeighborsClassifier(n_neighbors=1).fit(X_iris, y_iris).predict(X_new)
pred_mm = KNeighborsClassifier(n_neighbors=1).fit(X_iris_mm, y_iris).predict(X_new_mm)

n_changed = (pred_cm != pred_mm).sum()
print(f"predictions that changed when switching cm -> mm: {n_changed} out of {len(pred_cm)}")

<div class="alert alert-info">

<b>Think about it.</b>

1. The flowers did not change, only the unit we wrote their length in. So why did any prediction
   change at all?
2. In millimeters, which of the two features dominates the distance, and why?
3. The next cell standardizes both versions of the data and prints the largest difference between
   them. What do you expect it to print, and why?
</div>

**Your answer:**




In [ ]:
from sklearn.preprocessing import StandardScaler

sc_cm = StandardScaler().fit(X_iris)
sc_mm = StandardScaler().fit(X_iris_mm)

print("max difference between standardized cm and mm data:",
      np.abs(sc_cm.transform(X_iris) - sc_mm.transform(X_iris_mm)).max())

### 4.3 Splitting, and a trap to avoid

Finally we split the abalone data into a training and a test set.

One detail matters more than it looks. The scaler must learn $\mu_j$ and $\sigma_j$ from the
**training set only**, and then be applied to the test set. If you standardize everything
together first, information about the test set (its mean and spread) leaks into the training
procedure, and your test error is no longer an honest estimate of performance on new data.

`fit` on train, `transform` on both. That is the habit `Pipeline` will automate for you later.

In [ ]:
y_ab = abalone_encoded['age'].values
X_ab = abalone_encoded.drop(columns=['age']).astype(float).values

X_train, X_test, y_train, y_test = train_test_split(
    X_ab, y_ab, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler().fit(X_train)        # learn mu and sigma from TRAIN only
X_train = scaler.transform(X_train)           # apply to train
X_test = scaler.transform(X_test)             # and to test

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(f"train mean ~ {X_train.mean():.2e}, train sd ~ {X_train.std():.3f}")

## 5. $k$-NN for regression

So far the output was nominal. When the output is real-valued ($y \in \mathbb{R}$) the task is a
**regression problem**. Predicting an abalone's ring count is one.

The nearest neighbor idea carries over directly, with the vote replaced by an average. For the
$k$ neighbors $N_k(\boldsymbol{x})$ of an instance $\boldsymbol{x}$:

$$ f(\boldsymbol{x}) = \frac{1}{k} \sum_{\boldsymbol{x}_i \in N_k(\boldsymbol{x})} y_i. $$

Note what happens at the extremes, exactly as with classification: $k = 1$ copies the nearest
neighbor's value, and $k = n$ predicts the global mean of the training set for *every* instance,
ignoring the features completely.

<div class="alert alert-success">

<b>EXERCISE 5.1</b>: <b>Implement <code>knn_regress(x, X_train, y_train, k)</code>, returning the
mean output of the <code>k</code> training instances nearest to <code>x</code>.</b>

This is your <code>knn_iris_predict</code> with the vote replaced by <code>.mean()</code>. Note
that features and outputs now arrive as two separate arrays, rather than one DataFrame.
</div>

In [ ]:
def knn_regress(x, X_train, y_train, k=5):
    # Euclidean distance from x to every training instance
    dist_euc = ...   # TODO

    # positions of the k nearest
    nn_ind = ...     # TODO

    # average their outputs
    return ...       # TODO

And the moment of truth: your 4 lines against the library's implementation, on all 140 test
snails.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

my_preds = np.array([knn_regress(x, X_train, y_train, k=5) for x in X_test])
sk_preds = KNeighborsRegressor(n_neighbors=5, metric='euclidean').fit(
    X_train, y_train).predict(X_test)

assert np.allclose(my_preds, sk_preds), "predictions differ - check your implementation"
print("Identical to scikit-learn on all", len(X_test), "test instances.")
print("You have now written k-NN for both classification and regression from scratch.")

## 6. Using scikit-learn: baselines, error metrics and choosing $k$

From here on we use scikit-learn's implementation, which is faster and handles the awkward cases.
Its API is uniform across essentially every model in the library: construct an estimator with its
settings, call `fit(X, y)` to learn, call `predict(X)` to use it. Learning lab 3's models is then
mostly a matter of changing one class name.

### How wrong is a regression model?

For regression the standard metric is the **mean squared error**, the average squared gap between
prediction and truth:

$$ \text{MSE} = \frac{1}{|\mathcal{D}|} \sum_{\boldsymbol{x}_i \in \mathcal{D}} (f(\boldsymbol{x}_i) - y_i)^2. $$

Squaring makes errors positive and penalizes large misses disproportionately. Its drawback is the
units: an MSE of 6.3 is in *squared rings*, which means nothing intuitively. Taking the square root
gives the **RMSE**, back in rings. Report that one to a biologist.

### Compared to what?

A number like "MSE = 6.3" is meaningless on its own. Always compare against a baseline that uses
no features at all: predicting the training mean for every snail
([`DummyRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)).
Any model that cannot beat it has learned nothing.

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

# NOTE the argument order: mean_squared_error(y_true, y_pred).
# It happens not to matter for MSE, but it does for most other metrics - build the habit.
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
mse_base = mean_squared_error(y_test, baseline.predict(X_test))

knn5 = KNeighborsRegressor(n_neighbors=5, metric='euclidean').fit(X_train, y_train)
mse_knn = mean_squared_error(y_test, knn5.predict(X_test))

print(f"baseline (predict the mean)   MSE = {mse_base:6.3f}   RMSE = {mse_base ** .5:.3f} rings")
print(f"5-nearest neighbors          MSE = {mse_knn:6.3f}   RMSE = {mse_knn ** .5:.3f} rings")

So the model is off by about 2.6 rings on animals ranging from 3 to 24 rings, against 3.2
rings for a model that ignores every measurement. Real, but not dramatic, and far more
informative than either number alone.

A model is always flattered by the data it was fitted on, so training error will usually sit below
test error and only the test number estimates performance on new snails. The next exercise shows
both at once.

$k$ is a **hyperparameter**, a setting you choose rather than something learned from the data.
Small $k$ gives a flexible model that follows the training data closely (the jagged $k=1$
boundary of section 2); large $k$ gives a smooth, rigid one. Let us watch both errors as $k$
varies.

<div class="alert alert-success">

<b>EXERCISE 6.2</b>: <b>Fit a $k$-NN regressor for every $k$ from 1 to 50 in steps of 3. Store the
training and test MSE for each, and plot both against $k$.</b>
</div>

In [ ]:
ks = range(1, 51, 3)
results_train, results_test = [], []

for k in ks:
    knn = ...     # TODO: instantiate with n_neighbors=k
    ...           # TODO: fit on the training data
    results_train.append(...)   # TODO: training MSE
    results_test.append(...)    # TODO: test MSE

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ks, results_train, 'o-', label='train MSE')
ax.plot(ks, results_test, 's-', label='test MSE')
ax.axhline(mse_base, color='gray', linestyle='--', label='baseline (predict the mean)')
ax.set_xlabel("number of neighbors $k$")
ax.set_ylabel("mean squared error")
ax.set_title("Model complexity and error")
ax.legend()
plt.show()

best = list(ks)[int(np.argmin(results_test))]
print(f"lowest test MSE at k = {best}  (MSE = {min(results_test):.3f})")

<div class="alert alert-info">

<b>Read the plot carefully. This one figure contains most of the course's central idea.</b>

1. What is the training MSE at $k = 1$? Verify it with the cell below, and explain the value you
   get. What is the nearest neighbor of a training instance?
2. What happens to the training error as $k$ grows, and what happens to the test error? If a model
   achieves zero error on its training data, is it a good model?
3. Describe the shape of the test curve. At roughly which $k$ is it lowest, and what is going
   wrong at each end?
4. Even at the largest $k$ plotted, the test curve is still far below the gray baseline. How
   large would $k$ have to be before the model matched the baseline exactly, and why?
5. If you increase the range of the figure to k=150 you see the test MSE outperform the train MSE, can you explain how this can happen?
</div>

In [ ]:
zero = mean_squared_error(y_train, KNeighborsRegressor(n_neighbors=1).fit(
    X_train, y_train).predict(X_train))
print(f"training MSE at k = 1: {zero}")

**Your answer:**




<div class="alert alert-warning">
<b>An important caveat about what we just did.</b>

We chose $k$ by looking at the test error, and that invalidates the test set as an estimate of
future performance. Having used it to make a decision, we can no longer claim it is data the model
knows nothing about. The plot above is legitimate for <i>understanding</i> how error depends on
$k$; it is not how you select $k$ in practice.

The proper procedure, splitting off a separate validation set or using cross-validation, is the
subject of <b>lab 5</b>. Until then, be aware that this shortcut is a shortcut.
</div>